# vLLM Prefix Caching — How Sub-Second Inference Works

This notebook demonstrates **prefix caching** in vLLM: the mechanism that lets a shared context (system prompt, documents, reviews) be computed once and reused across thousands of requests.

**Runtime**: T4 GPU (free Colab tier is sufficient)

**Model**: `Qwen/Qwen2.5-7B-Instruct-AWQ` (~4 GB, fits on T4)

## 1. The Concept

Every LLM request has two phases:

| Phase | What happens | Cost |
|-------|-------------|------|
| **Prefill** | Compute KV pairs for every input token | O(n²) in context length — **slow** |
| **Decode** | Generate output tokens one by one | O(n) per token — fast |

Prefix caching skips prefill for any token sequence already seen. vLLM splits the context into **blocks of 16 tokens**, hashes each block (chained with the previous hash), and stores the computed KV state. On the next request, matching hashes = free prefill.

```
block 1: hash([system_prompt tokens])        → h1
block 2: hash([shared_context tokens] + h1)  → h2  ← reused if identical
block 3: hash([user_query tokens]    + h2)   → h3  ← always computed fresh
```

## 2. The Fashion Example

Imagine a retailer's sizing assistant. Four shoppers left these reviews for a structured blazer:

> - *"fits my waist but I can't button it after lunch"*
> - *"bought both M and L — the M looks better but L is more comfortable"*
> - *"first time buying this brand, not sure about European sizing"*
> - *"I'm between sizes and this is a structured blazer"*

Now three customers query the assistant with different questions. The system prompt + 4 reviews are **identical** across all three requests.

```
Alice: "Should I size up?"          ← request 1: full prefill (slow)
Bob:   "Is this true to size?"       ← request 2: cache hit   (fast)
Carol: "I'm a 38 EU, what to order?" ← request 3: cache hit   (fast)
```

Alice pays the prefill cost once. Bob and Carol get that computation for free.

**What defeats the cache:**
- Shuffling the review order (same words, different token positions → different hashes)
- Injecting a dynamic timestamp or session ID before the shared context
- Putting user-specific context *before* the shared context instead of *after*

In [ ]:
import vllm, sys
print(f">>> vllm {vllm.__version__}  python {sys.version.split()[0]}")
print(">>> Runtime restart required before the next cell — Runtime > Restart runtime")

In [ ]:
MODEL_HF      = "Qwen/Qwen2.5-7B-Instruct-AWQ"
GPU_MEM_UTIL  = 0.85
MAX_MODEL_LEN = 4096

In [ ]:
import time
import textwrap
from vllm import LLM, SamplingParams

In [ ]:
llm = LLM(
    model=MODEL_HF,
    enable_prefix_caching=True,
    enforce_eager=True,
    gpu_memory_utilization=GPU_MEM_UTIL,
    max_model_len=MAX_MODEL_LEN,
)

sampling = SamplingParams(temperature=0, max_tokens=80)

## 3. Experiment 1 — Cold Cache vs Warm Cache

We send the same shared context (system prompt + 4 reviews) with three different questions.
The first request is cold — it pays full prefill. The next two are warm — cache hits.

In [ ]:
SYSTEM = "You are a sizing assistant. Answer in one sentence based only on the reviews provided."

REVIEWS = """\
Reviews for Structured Blazer SKU-1042:
- "fits my waist but I can't button it after lunch"
- "bought both M and L — the M looks better but L is more comfortable"
- "first time buying this brand, not sure about European sizing"
- "I'm between sizes and this is a structured blazer"
"""

QUERIES = [
    "Should I size up?",
    "Is this true to size?",
    "I'm a 38 EU, what should I order?",
]

def make_prompt(query):
    return f"<|im_start|>system\n{SYSTEM}\n\n{REVIEWS}<|im_end|>\n<|im_start|>user\n{query}<|im_end|>\n<|im_start|>assistant\n"

print(f"Shared prefix length: {len(llm.get_tokenizer().encode(f'<|im_start|>system\n{SYSTEM}\n\n{REVIEWS}<|im_end|>'))} tokens")
print()

for i, query in enumerate(QUERIES):
    prompt = make_prompt(query)
    t0 = time.perf_counter()
    output = llm.generate([prompt], sampling)
    elapsed = time.perf_counter() - t0

    label = "COLD (full prefill)" if i == 0 else "WARM (cache hit)   "
    answer = output[0].outputs[0].text.strip()
    print(f"[{label}] {elapsed:.2f}s — Q: {query}")
    print(f"  A: {textwrap.shorten(answer, 120)}")
    print()

## 4. Experiment 2 — What Defeats the Cache

Same reviews, same questions — but now we **shuffle the review order** per request.
The token sequence changes → different hashes → every request is a cache miss.

In [ ]:
import random

RAW_REVIEWS = [
    '"fits my waist but I can\'t button it after lunch"',
    '"bought both M and L — the M looks better but L is more comfortable"',
    '"first time buying this brand, not sure about European sizing"',
    '"I\'m between sizes and this is a structured blazer"',
]

def make_prompt_shuffled(query):
    shuffled = RAW_REVIEWS[:]
    random.shuffle(shuffled)
    reviews = "Reviews for Structured Blazer SKU-1042:\n" + "\n".join(f"- {r}" for r in shuffled)
    return f"<|im_start|>system\n{SYSTEM}\n\n{reviews}\n<|im_end|>\n<|im_start|>user\n{query}<|im_end|>\n<|im_start|>assistant\n"

print("=== Shuffled reviews — every request is a cache miss ===")
print()
for i, query in enumerate(QUERIES):
    prompt = make_prompt_shuffled(query)
    t0 = time.perf_counter()
    output = llm.generate([prompt], sampling)
    elapsed = time.perf_counter() - t0
    print(f"[MISS] {elapsed:.2f}s — Q: {query}")

print()
print("Compare: all three requests take ~the same time as the cold request above.")

## 5. Experiment 3 — Dynamic Content Placement

A common mistake: injecting a session ID or timestamp **before** the shared context.
This shifts all subsequent token positions → hash mismatch from block 1 onward.

In [ ]:
import uuid

def make_prompt_session_prefix(query):
    """BAD: session ID before shared context — defeats cache."""
    session = uuid.uuid4().hex[:8]
    return f"<|im_start|>system\n[session:{session}]\n{SYSTEM}\n\n{REVIEWS}<|im_end|>\n<|im_start|>user\n{query}<|im_end|>\n<|im_start|>assistant\n"

def make_prompt_session_suffix(query):
    """GOOD: session ID after shared context — cache still hits on shared prefix."""
    session = uuid.uuid4().hex[:8]
    return f"<|im_start|>system\n{SYSTEM}\n\n{REVIEWS}<|im_end|>\n<|im_start|>user\n[session:{session}] {query}<|im_end|>\n<|im_start|>assistant\n"

query = QUERIES[0]

# Warm the cache first with a clean request
llm.generate([make_prompt(query)], sampling)

t0 = time.perf_counter()
llm.generate([make_prompt_session_prefix(query)], sampling)
t_bad = time.perf_counter() - t0

t0 = time.perf_counter()
llm.generate([make_prompt_session_suffix(query)], sampling)
t_good = time.perf_counter() - t0

print(f"Session ID BEFORE shared context (cache miss): {t_bad:.2f}s")
print(f"Session ID AFTER  shared context (cache hit):  {t_good:.2f}s")
print(f"Speedup from correct placement: {t_bad/t_good:.1f}x")

## 6. KV Cache Quantization — Fitting More Contexts

Cache eviction (LRU) happens when VRAM fills up. KV quantization delays eviction by shrinking each cached entry.

The math for one token's KV entry in Qwen2.5-7B:
- 28 layers × 2 (K+V) × 8 heads × 128 head_dim = 57,344 values per token
- FP16: × 2 bytes = **112 KB per token**
- INT4: × 0.5 bytes = **28 KB per token** (4× smaller)

In [ ]:
# Qwen2.5-7B architecture constants
LAYERS    = 28
KV_HEADS  = 8
HEAD_DIM  = 128

values_per_token = LAYERS * 2 * KV_HEADS * HEAD_DIM

fp16_bytes = values_per_token * 2
int4_bytes = values_per_token * 0.5

# Shared context size from our fashion example
context_tokens = 120

# Available KV cache VRAM on T4 (model uses ~4GB, leaving ~11GB)
kv_vram_gb = 11
kv_vram_bytes = kv_vram_gb * 1e9

contexts_fp16 = kv_vram_bytes / (fp16_bytes * context_tokens)
contexts_int4 = kv_vram_bytes / (int4_bytes * context_tokens)

print(f"KV entry size per token:")
print(f"  FP16 : {fp16_bytes/1024:.1f} KB/token")
print(f"  INT4 : {int4_bytes/1024:.1f} KB/token")
print()
print(f"Cached product contexts ({context_tokens} tokens each) in {kv_vram_gb}GB KV VRAM:")
print(f"  FP16 : {contexts_fp16:,.0f} contexts")
print(f"  INT4 : {contexts_int4:,.0f} contexts  (4× more — 4× fewer cache evictions)")

## Summary

| Mechanism | Effect |
|-----------|--------|
| Prefix caching | Skip O(n²) prefill for shared context — reuse cached KV blocks |
| Hash-based matching | Exact token-level match required — one token difference = cache miss |
| Stable prefix placement | Put shared context first, user-specific content last |
| LRU eviction | High-traffic contexts stay warm; long-tail contexts pay full prefill |
| KV quantization (INT4) | 4× more contexts fit before eviction — 4× better hit rate on large catalogues |

**The sub-second path**: small model (7B AWQ) + fits entirely in VRAM + shared context already cached = only the user's unique tokens need computation.